# DPLQR: epochs, coefficient drift, and prediction error
Created 08 September 2026 (08092026).

This **paired first-pass diagnostic** follows the six saved DPLQR datasets: Cases 1–3,
two repetitions, n=1000, tau=0.5, and an 800/200 training/validation split. Only training
duration and checkpoint selection vary. Architecture, learning rate, batches, data,
initialization, sparse masks, and stochastic training sequence stay fixed within a trajectory.
The parent notebook and its commented full-replication settings remain unchanged.

**Run All** with the repository `.venv` kernel. No separate simulation `.py` or R step is required.
This notebook imports selected scientific definitions from the parent notebook and the original
`dqAux.py`, without executing the parent's settings, simulation, or R cells.

Three rules are compared at each budget: **terminal epoch** (force training to that epoch),
**validation best** (train to the budget, restore the minimum validation-loss epoch), and
**patience 15** (the first pass's stopping/best-epoch rule). The first two deliberately continue
beyond early stopping to expose drift. They are diagnostic variants of the first-pass fit.

Two repetitions cannot establish population bias or general robustness. True theta and test
data are used only for diagnostics, never to choose epochs. No standard errors or coverage are fitted.

## 1. Imports and reused scientific definitions

In [ ]:
from __future__ import annotations
import ast
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
from scipy.stats import norm, t
from sklearn.preprocessing import StandardScaler
import torch
from torchtuples import Model
import torchtuples as tt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p/'dqAux.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open inside the dplqr repository with its .venv kernel.')
PARENT_DIR = ROOT/'results'/'2026-09-04-homoscedastic-simulation'
FIRST_PASS = PARENT_DIR/'first-pass-run'
EXPERIMENT_DIR = FIRST_PASS/'2026-09-08-epoch-robustness'
SOURCE_NOTEBOOK = PARENT_DIR/'simulate_homoscedastic.ipynb'
THIS_NOTEBOOK = EXPERIMENT_DIR/'epoch_robustness.ipynb'
sys.path.insert(0, str(ROOT))
from dqAux import dqNetSparse, checkLoss, checkErrorMean
THETA = np.array([1.0, -1.0])

# Import definitions only; no parent parameter, simulation, or R cells execute.
REUSED_NAMES = {'seed_all', 'nonlinear_truth', 'generate_covariates', 'generate_dataset',
                'as_numpy', 'clip_neural_weights', 'tensor_pair'}
selected_nodes = {}
for cell in nbformat.read(SOURCE_NOTEBOOK, as_version=4).cells:
    if cell.cell_type == 'code':
        for node in ast.parse(cell.source).body:
            if isinstance(node, ast.FunctionDef) and node.name in REUSED_NAMES:
                if node.name in selected_nodes:
                    raise ValueError('Duplicate scientific definition: '+node.name)
                selected_nodes[node.name] = node
assert set(selected_nodes) == REUSED_NAMES, 'Missing parent scientific definitions'
for node in selected_nodes.values():
    module = ast.fix_missing_locations(ast.Module(body=[node], type_ignores=[]))
    exec(compile(module, str(SOURCE_NOTEBOOK), 'exec'), globals())
reference_config = json.loads((FIRST_PASS/'run_config.json').read_text(encoding='utf-8'))
reference_raw = pd.read_csv(FIRST_PASS/'raw_results.csv', float_precision='round_trip')
reference_dplqr = reference_raw.loc[reference_raw.method.eq('DPLQR')].copy()
assert reference_config['status'] == 'complete', 'Complete the first pass first'
print('Python:', sys.executable)
print('Reused:', ', '.join(sorted(REUSED_NAMES)))

## 2. Independent epoch settings
The parent's active/full-replication profile is untouched. Changed epoch settings require a fresh
`OUTPUT_DIR`; unchanged completed trajectories are reused. Save edits and restart the kernel before
Run All so the source fingerprint corresponds to the executed code.

In [ ]:
EPOCH_BUDGETS = [10, 25, 50, 100, 200, 500, 1000]
PROFILE = dict(repetitions=2, cases=[1, 2, 3], n=1000, test_size=1000, tau=0.5,
               seed=20260904, depth=2, width=32, batch_size=128,
               learning_rate=0.005, patience=15, threads=1)
OUTPUT_DIR = EXPERIMENT_DIR
assert EPOCH_BUDGETS == sorted(set(EPOCH_BUDGETS)) and min(EPOCH_BUDGETS) >= 1
assert 100 in EPOCH_BUDGETS and max(EPOCH_BUDGETS) >= 100
for name in ('repetitions', 'test_size', 'seed', 'depth', 'width', 'batch_size',
             'learning_rate', 'patience', 'threads', 'cases'):
    assert PROFILE[name] == reference_config[name], f'{name} must match the saved paired baseline'
assert reference_config['sample_sizes'] == [PROFILE['n']]
assert reference_config['taus'] == [PROFILE['tau']]
assert reference_config['epochs'] == 100 and reference_config['hyperparameter_mode'] == 'fixed'
torch.set_num_threads(PROFILE['threads'])
torch.use_deterministic_algorithms(True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Epoch budgets:', EPOCH_BUDGETS)
print('Fixed paired profile:', PROFILE)
print('Outputs:', OUTPUT_DIR)

## 3. Verify the same datasets
The original generator/seed order also recover validation observations, absent from the R exports.
Every training/test value is checked against the existing exports before fitting. Training data alone
determine standardization. Network seeds match the saved first-pass fits.

In [ ]:
def paired_data(case, rep):
    n = PROFILE['n']
    setting_seed = PROFILE['seed'] + case*10_000_000 + n*1_000 + rep
    rng = np.random.default_rng(setting_seed)
    x, z, y, _ = generate_dataset(n, case, rng)
    order = rng.permutation(n)
    tr, va = order[:int(0.8*n)], order[int(0.8*n):]
    xt, zt, yt, mt = generate_dataset(PROFILE['test_size'], case, rng)
    stem = f'case_{case}_n_{n}_rep_{rep:04d}'
    checks = []
    for split, expected in [
        ('train', np.column_stack((y[tr], x[tr], z[tr], nonlinear_truth(z[tr], case)))),
        ('test', np.column_stack((yt, xt, zt, mt)))]:
        path = FIRST_PASS/'data_for_r'/f'{stem}_{split}.csv.gz'
        saved = pd.read_csv(path, float_precision='round_trip').to_numpy()
        np.testing.assert_allclose(saved, expected, rtol=0, atol=1e-12)
        checks.append(dict(case=case, rep=rep, split=split,
                           max_abs_difference=float(np.max(np.abs(saved-expected))),
                           file_sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    return dict(xtr=x[tr], ztr=z[tr], ytr=y[tr], xv=x[va], zv=z[va], yv=y[va],
                xt=xt, zt=zt, yt=yt, true_m=mt+t.ppf(PROFILE['tau'], df=3),
                network_seed=setting_seed+int(round(PROFILE['tau']*1_000_000))), checks

## 4. Observe one uninterrupted training path
Construction, initialization, loss, optimizer and `fit` arguments follow the parent's
`train_dplqr_once`. The observer never stops or modifies the training network. It clips an **evaluation
clone** for endpoint diagnostics, matching the parent's final clipping. Direct forward passes avoid
prediction DataLoaders, which can consume RNG and change subsequent training.

`selection_val_loss` preserves the original torchtuples **pre-clipping minibatch-averaged** validation
score for selecting epochs. Diagnostic train/validation/test losses use full-sample means after
clipping. They need not equal the selection score. Epoch 0 records initialization and is not selectable.

In [ ]:
class EpochRecorder(tt.callbacks.Callback):
    def __init__(self, evaluator, inputs, outcomes, true_m, test_x, case, rep):
        self.evaluator, self.inputs, self.outcomes = evaluator, inputs, outcomes
        self.true_m, self.test_x = true_m, test_x
        self.case, self.rep = case, rep
        self.rows = []

    def observe(self, epoch, score):
        rng_before = torch.random.get_rng_state().clone()
        self.evaluator.load_state_dict(self.model.net.state_dict())
        clip_neural_weights(self.evaluator)
        self.evaluator.eval()
        with torch.no_grad():
            theta = self.evaluator.linLinear.weight.detach().cpu().numpy().reshape(-1).copy()
            predictions = {name: self.evaluator(*pair).cpu().numpy().reshape(-1)
                           for name, pair in self.inputs.items()}
        assert torch.equal(rng_before, torch.random.get_rng_state()), 'Observer changed Torch RNG'
        losses = {name+'_check_loss': float(checkErrorMean(pred[:, None], self.outcomes[name][:, None],
                                                           tau=PROFILE['tau']))
                  for name, pred in predictions.items()}
        m_hat = predictions['test'] - self.test_x @ theta
        row = dict(case=self.case, n=PROFILE['n'], rep=self.rep, tau=PROFILE['tau'], epoch=epoch,
                   theta1=float(theta[0]), theta2=float(theta[1]),
                   error_theta1=float(theta[0]-THETA[0]), error_theta2=float(theta[1]-THETA[1]),
                   selection_val_loss=score,
                   relative_mse=float(np.mean((m_hat-self.true_m)**2)/np.mean(self.true_m**2)), **losses)
        assert np.isfinite([v for k,v in row.items() if k != 'selection_val_loss']).all()
        assert epoch == 0 or np.isfinite(score)
        self.rows.append(row)

    def on_fit_start(self):
        self.observe(0, np.nan)

    def on_epoch_end(self):
        epoch = len(self.rows)
        self.observe(epoch, float(self.model.val_metrics.scores['loss']['score'][-1]))
        if epoch % 250 == 0:
            print(f'case={self.case}, rep={self.rep}, epoch={epoch}', flush=True)
        return False

def train_epoch_path(data, case, rep):
    seed_all(data['network_seed'])
    scaler = StandardScaler().fit(data['ztr'])
    inputs = {'train': tensor_pair(data['xtr'], scaler.transform(data['ztr'])),
              'validation': tensor_pair(data['xv'], scaler.transform(data['zv'])),
              'test': tensor_pair(data['xt'], scaler.transform(data['zt']))}
    net = dqNetSparse(2, 8, torch.zeros((1, 2), dtype=torch.float32),
                      [PROFILE['depth'], PROFILE['width']], sparseRatio=0.5)
    net.linLinear.reset_parameters()
    model = Model(net, checkLoss(tau=PROFILE['tau']), device='cpu')
    model.optimizer.set_lr(PROFILE['learning_rate'])
    observer = EpochRecorder(copy.deepcopy(net).requires_grad_(False), inputs,
                             {'train': data['ytr'], 'validation': data['yv'], 'test': data['yt']},
                             data['true_m'], data['xt'], case, rep)
    model.fit(inputs['train'], torch.tensor(data['ytr'][:, None], dtype=torch.float32),
              PROFILE['batch_size'], max(EPOCH_BUDGETS), [observer], False,
              val_data=(inputs['validation'], torch.tensor(data['yv'][:, None], dtype=torch.float32)),
              val_batch_size=PROFILE['batch_size'])
    frame = pd.DataFrame(observer.rows)
    assert frame.epoch.tolist() == list(range(max(EPOCH_BUDGETS)+1))
    return frame

def choose_epoch(trace, budget, rule):
    eligible = trace.loc[trace.epoch.between(1, budget)].sort_values('epoch')
    if rule == 'terminal':
        return eligible.iloc[-1], budget
    best, since_best, stopped_at = None, 0, budget
    for row in eligible.itertuples(index=False):
        if best is None or row.selection_val_loss < best.selection_val_loss:
            best, since_best = row, 0
        else:
            since_best += 1
        if rule == 'patience15' and since_best >= PROFILE['patience']:
            stopped_at = row.epoch
            break
    return trace.loc[trace.epoch.eq(best.epoch)].iloc[0], int(stopped_at)

def atomic_csv(frame, path):
    temporary = path.with_suffix('.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def experiment_identity():
    nb = nbformat.read(THIS_NOTEBOOK, as_version=4)
    source = '\n'.join(ast.dump(ast.parse(c.source), include_attributes=False)
                        for c in nb.cells if c.cell_type == 'code')
    reused = {name: ast.dump(node, include_attributes=False) for name, node in selected_nodes.items()}
    return dict(profile=PROFILE, budgets=EPOCH_BUDGETS,
                code_sha256=hashlib.sha256(source.encode()).hexdigest(),
                scientific_functions_sha256=hashlib.sha256(json.dumps(reused, sort_keys=True).encode()).hexdigest(),
                dqAux_sha256=hashlib.sha256((ROOT/'dqAux.py').read_bytes()).hexdigest(),
                baseline_config_sha256=hashlib.sha256((FIRST_PASS/'run_config.json').read_bytes()).hexdigest(),
                baseline_results_sha256=hashlib.sha256((FIRST_PASS/'raw_results.csv').read_bytes()).hexdigest(),
                versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                              torch=torch.__version__, torchtuples=tt.__version__))

## 5. Run and reproduce the first-pass baseline
Completed per-repetition CSVs checkpoint six paths. The reconstructed cap-100/patience-15 coefficients,
nuisance error, and stopping epochs must match the saved first pass. Failed checks stop analysis.
Prediction tolerances allow float32 batched versus whole-sample evaluation.

In [ ]:
def run_experiment():
    identity = experiment_identity()
    config_path = OUTPUT_DIR/'epoch_run_config.json'
    trajectories_dir = OUTPUT_DIR/'trajectories'
    trajectories_dir.mkdir(exist_ok=True)
    if config_path.exists():
        previous = json.loads(config_path.read_text(encoding='utf-8'))
        if previous['identity'] != identity:
            raise ValueError('Settings/source changed. Choose a fresh OUTPUT_DIR.')
    elif list(trajectories_dir.glob('*.csv')):
        raise ValueError('Existing trajectories lack configuration. Choose a fresh OUTPUT_DIR.')
    started = time.perf_counter()
    metadata = dict(created_utc=datetime.now(timezone.utc).isoformat(), identity=identity, status='running',
                    completed_trajectories=0, expected_trajectories=6,
                    early_stopping='Virtual only; live training continues to the maximum epoch',
                    inference='No SEs, auxiliary projection fits, R calls, or coverage')
    traces, data_checks, baseline_checks, budget_rows = [], [], [], []
    config_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    try:
        for case, rep in itertools.product(PROFILE['cases'], range(1, PROFILE['repetitions']+1)):
            data, checks = paired_data(case, rep)
            data_checks.extend(checks)
            path = trajectories_dir/f'case_{case}_rep_{rep:04d}.csv'
            if path.exists():
                trace = pd.read_csv(path, float_precision='round_trip')
                assert trace.epoch.tolist() == list(range(max(EPOCH_BUDGETS)+1))
                assert trace.case.eq(case).all() and trace.rep.eq(rep).all()
                assert np.isfinite(trace.drop(columns='selection_val_loss').to_numpy()).all()
                assert np.isfinite(trace.loc[trace.epoch.gt(0), 'selection_val_loss']).all()
                print(f'Reusing case={case}, rep={rep}', flush=True)
            else:
                trace = train_epoch_path(data, case, rep)
                atomic_csv(trace, path)
            baseline, stop = choose_epoch(trace, 100, 'patience15')
            saved = reference_dplqr.loc[reference_dplqr.case.eq(case) & reference_dplqr.rep.eq(rep)].iloc[0]
            np.testing.assert_allclose(baseline[['theta1','theta2']].to_numpy(dtype=float),
                                       saved[['theta1','theta2']].to_numpy(dtype=float), rtol=0, atol=1e-7)
            np.testing.assert_allclose(baseline.relative_mse, saved.rmse_m, rtol=0, atol=1e-6)
            np.testing.assert_allclose(baseline.test_check_loss, saved.test_check_loss, rtol=0, atol=1e-6)
            assert stop == saved.epochs_run
            baseline_checks.append(dict(case=case, rep=rep, selected_epoch=int(baseline.epoch),
                actual_stop_epoch=stop, theta1_difference=float(baseline.theta1-saved.theta1),
                theta2_difference=float(baseline.theta2-saved.theta2),
                relative_mse_difference=float(baseline.relative_mse-saved.rmse_m), passed=True))
            for budget, rule in itertools.product(EPOCH_BUDGETS, ['terminal','validation_best','patience15']):
                selected, stopped_at = choose_epoch(trace, budget, rule)
                row = selected.to_dict()
                row.pop('epoch')
                row.update(epoch_budget=budget, rule=rule, selected_epoch=int(selected.epoch), epochs_trained=stopped_at)
                budget_rows.append(row)
            traces.append(trace)
            metadata.update(completed_trajectories=len(traces), elapsed_seconds=time.perf_counter()-started)
            config_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        raw, budgets = pd.concat(traces, ignore_index=True), pd.DataFrame(budget_rows)
        for frame in (raw, budgets):
            for name in ('case','n','rep'):
                frame[name] = frame[name].astype(int)
        assert len(raw) == 6*(max(EPOCH_BUDGETS)+1)
        assert len(budgets) == 6*len(EPOCH_BUDGETS)*3
        for frame, name in [(raw,'epoch_trajectories'), (budgets,'epoch_budget_results'),
                            (pd.DataFrame(data_checks),'data_pairing_checks'),
                            (pd.DataFrame(baseline_checks),'baseline_reproduction_checks')]:
            atomic_csv(frame, OUTPUT_DIR/f'{name}.csv')
        metadata.update(status='complete', trajectory_rows=len(raw), budget_rows=len(budgets),
                        baseline_verified=True, elapsed_seconds=time.perf_counter()-started)
        config_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        return raw, budgets
    except Exception as exc:
        metadata.update(status='failed', error=repr(exc), elapsed_seconds=time.perf_counter()-started)
        config_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        raise

trajectories, budget_results = run_experiment()
display(pd.read_csv(OUTPUT_DIR/'baseline_reproduction_checks.csv'))

## 6. Summary and paired drift tables
Signed bias is mean(theta_hat − theta) over **two** repetitions; SD uses ddof=1. Mean absolute error
reveals distances hidden by sign cancellation. A positive absolute-error change indicates movement
farther from truth for that dataset. Nuisance relative MSE uses no square root, following the paper.

In [ ]:
summary_rows = []
for (case,budget,rule), group in budget_results.groupby(['case','epoch_budget','rule']):
    row = dict(case=case, n=PROFILE['n'], tau=PROFILE['tau'], epoch_budget=budget, rule=rule,
               repetitions=len(group), mean_selected_epoch=group.selected_epoch.mean(),
               mean_relative_mse=group.relative_mse.mean(), mean_train_check_loss=group.train_check_loss.mean(),
               mean_validation_check_loss=group.validation_check_loss.mean(),
               mean_test_check_loss=group.test_check_loss.mean())
    for j in (1,2):
        error = group[f'theta{j}']-THETA[j-1]
        row.update({f'mean_theta{j}':group[f'theta{j}'].mean(), f'bias_theta{j}':error.mean(),
                    f'sd_theta{j}':group[f'theta{j}'].std(ddof=1),
                    f'mae_theta{j}':error.abs().mean(), f'mse_theta{j}':(error**2).mean()})
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR/'epoch_summary.csv', index=False)
coefficient_table = summary[['case','n','tau','rule','epoch_budget']].copy()
for j in (1,2):
    coefficient_table[f'theta{j}_bias_sd'] = [f'{b:.4f} ({s:.4f})' for b,s in
                                             zip(summary[f'bias_theta{j}'],summary[f'sd_theta{j}'])]
coefficient_table['relative_mse'] = summary.mean_relative_mse
coefficient_table.to_csv(OUTPUT_DIR/'coefficient_bias_sd_by_epoch.csv',index=False)
terminal = budget_results.loc[budget_results.rule.eq('terminal')]
start = terminal.loc[terminal.epoch_budget.eq(100)].set_index(['case','rep'])
end = terminal.loc[terminal.epoch_budget.eq(max(EPOCH_BUDGETS))].set_index(['case','rep'])
drift = start[['n','tau']].copy()
for j in (1,2):
    drift[f'theta{j}_at_100'] = start[f'theta{j}']
    drift[f'theta{j}_at_final'] = end[f'theta{j}']
    drift[f'theta{j}_change'] = end[f'theta{j}']-start[f'theta{j}']
    drift[f'absolute_error_change_theta{j}'] = (end[f'theta{j}']-THETA[j-1]).abs()-(start[f'theta{j}']-THETA[j-1]).abs()
drift['relative_mse_change'] = end.relative_mse-start.relative_mse
drift['test_check_loss_change'] = end.test_check_loss-start.test_check_loss
drift = drift.reset_index()
drift.to_csv(OUTPUT_DIR/'paired_drift_100_to_final.csv',index=False)
display(coefficient_table.loc[coefficient_table.epoch_budget.isin([100,500,1000])])
display(drift)

## 7. Figures
Individual paths and two-repetition summaries are shown without confidence bands.
PNG and vector PDF figures are saved in `figures/`. The vertical reference is 100 epochs.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter

# Paired exploratory comparisons: the same two samples and initializations are
# used at every budget. The two repetitions do not support confidence bands.
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "semibold", "axes.labelcolor": "#263238",
    "text.color": "#263238", "axes.edgecolor": "#9aa4aa",
    "grid.color": "#dbe1e5", "grid.linewidth": 0.6,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
case_order = list(PROFILE["cases"])
rep_order = sorted(trajectories["rep"].unique())
rep_colors = {rep: color for rep, color in zip(rep_order, ["#2878b5", "#d97732"])}
rule_styles = {
    "terminal": ("#2878b5", "-", "o", "At the final epoch"),
    "validation_best": ("#7b55a3", "--", "s", "Best validation epoch within budget"),
    "patience15": ("#168375", "-.", "^", "Early stopping: patience 15"),
}
diagnostic_curves = trajectories.loc[trajectories["epoch"] >= 1].copy()
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
figure_paths = []


def epoch_axis(ax, budgets_only=False):
    ax.axvline(100, color="#939a9e", linestyle=":", linewidth=0.8)
    ax.set_xscale("log")
    ticks = list(EPOCH_BUDGETS)
    if not budgets_only and 1 not in ticks:
        ticks = [1] + ticks
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xlim(min(ticks), max(ticks))
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(True, which="major", alpha=0.75)
    ax.set_axisbelow(True)


def save_diagnostic(fig, stem):
    fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")
    figure_paths.extend([FIGURE_DIR / f"{stem}.png", FIGURE_DIR / f"{stem}.pdf"])
    display(fig)
    plt.close(fig)


fig, axes = plt.subplots(len(case_order), 2, figsize=(12.5, 9.6), squeeze=False)
for row_index, case in enumerate(case_order):
    case_data = diagnostic_curves.loc[diagnostic_curves["case"] == case]
    for column, component in enumerate((1, 2)):
        ax = axes[row_index, column]
        for rep in rep_order:
            path = case_data.loc[case_data["rep"] == rep].sort_values("epoch")
            baseline = reference_dplqr.loc[
                (reference_dplqr["case"] == case) & (reference_dplqr["rep"] == rep)
            ]
            if len(baseline) != 1:
                raise ValueError(f"Expected one first-pass baseline for case {case}, repetition {rep}")
            ax.plot(path["epoch"], path[f"theta{component}"],
                    color=rep_colors[rep], linewidth=1.15, alpha=0.9)
            ax.axhline(float(baseline.iloc[0][f"theta{component}"]),
                       color=rep_colors[rep], linestyle=":", linewidth=1.4, alpha=0.9)
        ax.axhline(float(THETA[column]), color="#222222", linestyle="--", linewidth=1.2)
        ax.set_title(f"Case {case} | $\\theta_{component}$", loc="left", fontsize=11)
        ax.set_ylabel("Coefficient estimate")
        epoch_axis(ax)
        if row_index == len(case_order) - 1:
            ax.set_xlabel("Completed training epochs (log scale)")
handles = [Line2D([0], [0], color=rep_colors[rep], label=f"Repetition {rep}") for rep in rep_order]
handles += [Line2D([0], [0], color="#222222", linestyle="--", label="True coefficient"),
            Line2D([0], [0], color="#666666", linestyle=":", label="First-pass selected estimate")]
fig.suptitle("Do the DPLQR coefficients drift as training continues?", fontsize=15, y=0.98)
fig.text(0.5, 0.945, "Paired trajectories with early stopping disabled; the dotted baselines retain the first-pass stopping rule.",
         ha="center", fontsize=10)
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.035), ncol=4, frameon=False)
fig.text(0.5, 0.012, "Exploratory only: two repetitions per case; n = 1,000, median quantile; true coefficients are 1 and -1.",
         ha="center", fontsize=9, color="#52636b")
fig.subplots_adjust(left=0.08, right=0.985, top=0.895, bottom=0.135, hspace=0.38, wspace=0.25)
save_diagnostic(fig, "theta_trajectories")


for statistic, ylabel, title, subtitle, stem in [
    ("bias", "Mean signed error", "Mean coefficient error by epoch budget",
     "Signed error is the average estimate minus the true coefficient; opposite errors can cancel.",
     "bias_by_epoch_and_rule"),
    ("mae", "Mean absolute error", "Coefficient accuracy by epoch budget",
     "Absolute errors avoid cancellation between the two repetitions; lower values indicate closer estimates.",
     "absolute_error_by_epoch_and_rule"),
]:
    fig, axes = plt.subplots(len(case_order), 2, figsize=(12.5, 9.6), squeeze=False)
    for row_index, case in enumerate(case_order):
        for column, component in enumerate((1, 2)):
            ax = axes[row_index, column]
            for rule, (color, linestyle, marker, label) in rule_styles.items():
                values = summary.loc[(summary["case"] == case) & (summary["rule"] == rule)].sort_values("epoch_budget")
                ax.plot(values["epoch_budget"], values[f"{statistic}_theta{component}"],
                        color=color, linestyle=linestyle, marker=marker, markersize=4,
                        linewidth=1.6, label=label)
            ax.axhline(0, color="#5d6770", linestyle=":", linewidth=1)
            if statistic == "mae":
                ax.set_ylim(bottom=0)
            ax.set_title(f"Case {case} | $\\theta_{component}$", loc="left", fontsize=11)
            ax.set_ylabel(ylabel)
            epoch_axis(ax, budgets_only=True)
            if row_index == len(case_order) - 1:
                ax.set_xlabel("Maximum epoch budget (log scale)")
    fig.suptitle(title, fontsize=15, y=0.98)
    fig.text(0.5, 0.945, subtitle, ha="center", fontsize=10)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.035), ncol=3, frameon=False)
    fig.text(0.5, 0.012, "Two-repetition exploratory means; these are not precise Monte Carlo bias or robustness estimates.",
             ha="center", fontsize=9, color="#52636b")
    fig.subplots_adjust(left=0.08, right=0.985, top=0.895, bottom=0.135, hspace=0.38, wspace=0.25)
    save_diagnostic(fig, stem)


fig, axes = plt.subplots(len(case_order), 2, figsize=(12.5, 10), squeeze=False)
loss_styles = {
    "train_check_loss": ("#2878b5", "-", "Training check loss"),
    "validation_check_loss": ("#d97732", "--", "Validation check loss"),
    "test_check_loss": ("#525c67", "-.", "Test check loss"),
}
for row_index, case in enumerate(case_order):
    averaged = diagnostic_curves.loc[diagnostic_curves["case"] == case].groupby("epoch", as_index=False).agg(
        train_check_loss=("train_check_loss", "mean"),
        validation_check_loss=("validation_check_loss", "mean"),
        test_check_loss=("test_check_loss", "mean"),
    )
    ax = axes[row_index, 0]
    for field, (color, linestyle, label) in loss_styles.items():
        ax.plot(averaged["epoch"], averaged[field], color=color, linestyle=linestyle,
                linewidth=1.3, label=label)
    ax.set_title(f"Case {case} | Prediction loss at each epoch", loc="left", fontsize=11)
    ax.set_ylabel("Mean check loss")
    epoch_axis(ax)
    ax = axes[row_index, 1]
    for rule, (color, linestyle, marker, label) in rule_styles.items():
        values = summary.loc[(summary["case"] == case) & (summary["rule"] == rule)].sort_values("epoch_budget")
        ax.plot(values["epoch_budget"], values["mean_relative_mse"],
                color=color, linestyle=linestyle, marker=marker, markersize=4,
                linewidth=1.6, label=label)
    ax.set_title(f"Case {case} | Nuisance-function error by budget", loc="left", fontsize=11)
    ax.set_ylabel("Mean relative MSE")
    ax.set_ylim(bottom=0)
    epoch_axis(ax, budgets_only=True)
    if row_index == len(case_order) - 1:
        axes[row_index, 0].set_xlabel("Completed training epochs (log scale)")
        axes[row_index, 1].set_xlabel("Maximum epoch budget (log scale)")
fig.suptitle("Does prediction improve while the coefficients move?", fontsize=15, y=0.98)
fig.text(0.5, 0.946, "Two-repetition means. Relative MSE uses the paper's squared-error ratio, without a square root.",
         ha="center", fontsize=10)
loss_handles, loss_labels = axes[0, 0].get_legend_handles_labels()
rule_handles, rule_labels = axes[0, 1].get_legend_handles_labels()
fig.legend(loss_handles, loss_labels, loc="lower center", bbox_to_anchor=(0.5, 0.059), ncol=3, frameon=False)
fig.legend(rule_handles, rule_labels, loc="lower center", bbox_to_anchor=(0.5, 0.032), ncol=3, frameon=False)
fig.text(0.5, 0.010, "Test loss and known truth are diagnostics only; neither is used to select the epoch.",
         ha="center", fontsize=9, color="#52636b")
fig.subplots_adjust(left=0.08, right=0.985, top=0.895, bottom=0.157, hspace=0.38, wspace=0.25)
save_diagnostic(fig, "prediction_diagnostics")
print(f"Saved {len(figure_paths)} figure files (four PNGs and four PDFs) to {FIGURE_DIR}")

## 8. Results report

In [ ]:
def markdown_table(frame):
    lines = ['| '+' | '.join(map(str,frame.columns))+' |',
             '| '+' | '.join(['---']*len(frame.columns))+' |']
    for values in frame.itertuples(index=False,name=None):
        lines.append('| '+' | '.join(map(str,values))+' |')
    return '\n'.join(lines)

last = max(EPOCH_BUDGETS)
labels = {'terminal':'Terminal epoch','validation_best':'Validation best','patience15':'Patience 15'}
rows = []
for case,rule,budget in itertools.product(PROFILE['cases'],labels,sorted(set([100,500,last]))):
    if budget not in EPOCH_BUDGETS: continue
    g = summary.loc[summary.case.eq(case)&summary.rule.eq(rule)&summary.epoch_budget.eq(budget)].iloc[0]
    rows.append({'Case':case,'Rule':labels[rule],'Epoch cap':budget,
                 'theta1 bias (SD)':f'{g.bias_theta1:.4f} ({g.sd_theta1:.4f})',
                 'theta2 bias (SD)':f'{g.bias_theta2:.4f} ({g.sd_theta2:.4f})',
                 'Nuisance relative MSE':f'{g.mean_relative_mse:.4f}'})
farther1 = int(drift.absolute_error_change_theta1.gt(0).sum())
farther2 = int(drift.absolute_error_change_theta2.gt(0).sum())
metadata = json.loads((OUTPUT_DIR/'epoch_run_config.json').read_text())
lines = ['# DPLQR epoch experiment: first-pass results','',
    'Created 08 September 2026 (08092026).','',
    f'Six paired trajectories through {last} epochs completed in {metadata["elapsed_seconds"]:.1f} seconds. '
    'Cases 1–3, n=1000, tau=0.5, two repetitions. The parent first-pass/full-replication settings are unchanged.','',
    '## Findings','',
    f'Between terminal epochs 100 and {last}, absolute theta1 error increased in **{farther1}/6** datasets '
    f'and absolute theta2 error increased in **{farther2}/6**. Signed errors can cancel across repetitions; '
    'use the individual paths and absolute-error changes to assess drift.','']
for case in PROFILE['cases']:
    a = summary.loc[summary.case.eq(case)&summary.rule.eq('terminal')&summary.epoch_budget.eq(100)].iloc[0]
    b = summary.loc[summary.case.eq(case)&summary.rule.eq('terminal')&summary.epoch_budget.eq(last)].iloc[0]
    lines.append(f'- Case {case}: mean theta1 {a.mean_theta1:.4f} → {b.mean_theta1:.4f}; '
                 f'theta1 bias {a.bias_theta1:.4f} → {b.bias_theta1:.4f}; '
                 f'mean absolute theta1 error {a.mae_theta1:.4f} → {b.mae_theta1:.4f}; '
                 f'nuisance relative MSE {a.mean_relative_mse:.4f} → {b.mean_relative_mse:.4f}.')
unchanged = 0
for case,rep in itertools.product(PROFILE['cases'],range(1,PROFILE['repetitions']+1)):
    g = budget_results.loc[budget_results.case.eq(case)&budget_results.rep.eq(rep)&budget_results.rule.eq('patience15')]
    unchanged += int(g.loc[g.epoch_budget.eq(100),'selected_epoch'].iloc[0] ==
                     g.loc[g.epoch_budget.eq(last),'selected_epoch'].iloc[0])
lines += ['',f'Keeping patience-15 stopping leaves the selected epoch unchanged from cap 100 to cap {last} '
          f'in **{unchanged}/6** datasets. Raising the cap differs from forcing longer training. '
          'Validation selection uses no true coefficients or test results.','',
          '## Figures','']
for title,name in [('Coefficient trajectories','theta_trajectories'),
                   ('Bias by epoch and rule','bias_by_epoch_and_rule'),
                   ('Absolute error by epoch and rule','absolute_error_by_epoch_and_rule'),
                   ('Prediction diagnostics','prediction_diagnostics')]:
    lines += [f'![{title}](figures/{name}.png)','']
lines += ['## Selected budgets: bias (sample SD) and nuisance error','',markdown_table(pd.DataFrame(rows)),'',
    'The saved first-pass baseline is **Patience 15 / cap 100**, not terminal epoch 100. '
    'True theta=(1,-1). Relative MSE follows the paper definition, with no square root.','',
    '## Interpretation and limits','',
    'This isolates duration and checkpoint selection at depth=2, width=32, learning rate=0.005 and '
    'batch size=128. More epochs can improve one coefficient/case while worsening another. Two repetitions '
    'do not establish general robustness, reliable Monte Carlo bias, or a reproduction of the paper. '
    'A full 200-repetition study with the paper tuning grid remains separate. The 1000-epoch terminal '
    'branch deliberately extends training beyond the first-pass stopping rule. No SEs or coverage are '
    'reported, so no R or auxiliary-network fits are needed.','',
    '## Reproduction checks and files','',
    '- Regenerated training/test data match saved exports within 1e-12.',
    '- All six cap-100/patience-15 fits match saved coefficients (1e-7), nuisance errors (1e-6), '
    'and exact stopping epochs.',
    '- The observer verified unchanged Torch RNG and clipped only an evaluation clone.',
    '- [Source notebook](epoch_robustness.ipynb) / [executed notebook](executed_epoch_robustness.ipynb).',
    '- [Epoch paths](epoch_trajectories.csv), [budget results](epoch_budget_results.csv), '
    '[summary](epoch_summary.csv), [bias and SD](coefficient_bias_sd_by_epoch.csv).',
    '- [Paired changes](paired_drift_100_to_final.csv), [baseline checks](baseline_reproduction_checks.csv), '
    '[configuration](epoch_run_config.json).',
    '- Four figures are saved in PNG and vector PDF format in `figures/`.','']
report = '\n'.join(lines)
(OUTPUT_DIR/'RESULTS.md').write_text(report,encoding='utf-8')
display(Markdown(report.split('## Figures')[0]))
print('Report:',OUTPUT_DIR/'RESULTS.md')